[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/02.%20Parte%202/15.%20Clase%2015/13Class%20NB.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jdsanch1/SimRC/master?labpath=02.%20Parte%202%2F15.%20Clase%2015%2F13Class%20NB.ipynb)

In [ ]:
import importlib, subprocess, sys

_required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "sklearn", "statsmodels", "cvxpy", "pyomo"]
_missing  = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

# Clase 15: Optimización convexa (CVXPY) y programación estocástica (Pyomo)

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

## Objetivos de aprendizaje

- Resolver un portafolio con **restricción de riesgo** usando CVXPY (QCQP).
- Formular un problema de **optimización robusta** bajo incertidumbre (SOCP).
- Entender la diferencia entre **CVXPY** (convexo, verificación automática) y **Pyomo** (general, variables enteras).
- Resolver el **problema de dieta** (MILP) con Pyomo como ejemplo de programación matemática general.
- Consolidar los conceptos de las 15 clases del curso.

---

## Introducción: CVXPY vs. Pyomo

### ¿Por qué dos herramientas?

| Característica | CVXPY | Pyomo |
|---------------|-------|-------|
| **Enfoque** | Optimización convexa (DCP) | Programación matemática general |
| **Variables** | Continuas | Continuas, enteras, binarias |
| **Verificación** | Verifica convexidad automáticamente | No verifica convexidad |
| **Solvers** | ECOS, SCS, MOSEK | GLPK, CBC, Gurobi, CPLEX |
| **Complejidad** | Polinomial (punto interior) | NP-hard para enteros (branch-and-bound) |
| **Uso ideal** | Portafolios, regresión, SVM | Planificación, scheduling, MILP |

> **Regla práctica**: si el problema es convexo con variables continuas → **CVXPY**. Si necesitas variables enteras o el problema no es convexo → **Pyomo** (Boyd & Vandenberghe, 2004, §4.1.3).

## 1. Portafolio con restricción de riesgo (CVXPY)

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import cvxpy as cp
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.covariance as skcov
import statsmodels.api as sm

sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 4)

In [ ]:
assets = ["AAPL", "MSFT", "AA", "AMZN", "KO", "QAI"]
data = yf.download(assets, start="2025-01-01", end="2025-03-27",
                   auto_adjust=True, progress=False)
closes = data["Close"].dropna()
daily_returns = np.log(closes / closes.shift(1)).dropna()

# Estimadores robustos
huber = sm.robust.scale.Huber()
mu = np.array([huber(daily_returns[col].values)[0] for col in daily_returns.columns])
Sigma = skcov.LedoitWolf().fit(daily_returns).covariance_
n = len(assets)

print(f"Activos: {assets}")
print(f"Observaciones: {len(daily_returns)}")

### Formulación QCQP (Boyd §4.4.2)

Maximizar rendimiento sujeto a una restricción cuadrática de riesgo:

$$
\max_{\mathbf{w}} \quad \boldsymbol{\mu}^\top \mathbf{w}
$$

sujeto a:

$$
\mathbf{w}^\top \Sigma \, \mathbf{w} \leq \sigma_{\max}^2, \qquad \sum_i w_i = 1, \qquad w_i \geq 0
$$

La restricción $\mathbf{w}^\top \Sigma \, \mathbf{w} \leq \sigma_{\max}^2$ define un **elipsoide** en el espacio de pesos (conjunto convexo cuando $\Sigma \succeq 0$). El problema es un QCQP convexo.

In [ ]:
# Resolver para varios niveles de riesgo máximo
risk_levels = np.linspace(0.00001, 0.0005, 50)
opt_returns = []
opt_weights_list = []

w = cp.Variable(n)
sigma_max = cp.Parameter(nonneg=True)
objective = cp.Maximize(mu @ w)
constraints = [cp.quad_form(w, Sigma) <= sigma_max, cp.sum(w) == 1, w >= 0]
prob = cp.Problem(objective, constraints)

for s in risk_levels:
    sigma_max.value = s
    prob.solve(solver=cp.ECOS, warm_start=True)
    if prob.status == "optimal":
        opt_returns.append(252 * prob.value)
        opt_weights_list.append(w.value.copy())
    else:
        opt_returns.append(np.nan)
        opt_weights_list.append(np.full(n, np.nan))

opt_risks = np.sqrt(252 * risk_levels[:len(opt_returns)])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(opt_risks, opt_returns, "b-o", markersize=3, label="Frontera (restricción de riesgo)")
ax.set_xlabel("Riesgo máximo (σ anualizada)")
ax.set_ylabel("Rendimiento esperado (anualizado)")
ax.set_title("Frontera eficiente via QCQP (maximizar rendimiento sujeto a riesgo)")
ax.legend()
plt.tight_layout()

### Asignación óptima para un nivel de riesgo específico

In [ ]:
# Elegir un nivel de riesgo intermedio
target_risk = 0.0001
sigma_max.value = target_risk
prob.solve(solver=cp.ECOS)

print(f"Restricción de riesgo: σ²_max = {target_risk}")
print(f"Status: {prob.status}")
print(f"Rendimiento esperado diario: {prob.value:.6f}")
print(f"Rendimiento anualizado: {252 * prob.value:.4f}")
print(f"\nAsignación óptima:")
for name, val in zip(assets, w.value):
    print(f"  {name}: {val:.4f}")

---
## 2. Optimización robusta bajo incertidumbre (SOCP)

### El problema de la incertidumbre en los parámetros

Los rendimientos esperados $\boldsymbol{\mu}$ son los parámetros más difíciles de estimar. La **optimización robusta** busca la mejor solución que funcione bien bajo **todos los escenarios** posibles de $\boldsymbol{\mu}$ dentro de un conjunto de incertidumbre (Boyd §7.1).

### Incertidumbre elipsoidal

Si $\boldsymbol{\mu}$ puede variar dentro de un elipsoide centrado en la estimación $\hat{\boldsymbol{\mu}}$:

$$
\boldsymbol{\mu} \in \mathcal{U} = \{\hat{\boldsymbol{\mu}} + \rho \, \mathbf{u} : \|\mathbf{u}\|_2 \leq 1\}
$$

donde $\rho > 0$ mide el **grado de incertidumbre**, el peor caso para el rendimiento es:

$$
\min_{\boldsymbol{\mu} \in \mathcal{U}} \boldsymbol{\mu}^\top \mathbf{w} = \hat{\boldsymbol{\mu}}^\top \mathbf{w} - \rho \|\mathbf{w}\|_2
$$

El problema robusto se convierte en un **SOCP** (convexo):

$$
\max_{\mathbf{w}} \quad \hat{\boldsymbol{\mu}}^\top \mathbf{w} - \rho \|\mathbf{w}\|_2 \qquad \text{s.a.} \quad \sum_i w_i = 1, \quad w_i \geq 0
$$

In [ ]:
# Comparar portafolios: nominal vs robusto
rho_values = [0, 0.0001, 0.0005, 0.001]

fig, ax = plt.subplots(figsize=(10, 6))

for rho in rho_values:
    w_rob = cp.Variable(n)
    if rho == 0:
        obj = cp.Maximize(mu @ w_rob)
    else:
        obj = cp.Maximize(mu @ w_rob - rho * cp.norm(w_rob, 2))
    cons = [cp.sum(w_rob) == 1, w_rob >= 0]
    prob_rob = cp.Problem(obj, cons)
    prob_rob.solve(solver=cp.ECOS)
    
    if prob_rob.status == "optimal":
        label = f"ρ = {rho}" + (" (nominal)" if rho == 0 else "")
        ax.bar(np.arange(n) + rho_values.index(rho) * 0.2, w_rob.value,
               width=0.18, label=label)

ax.set_xticks(np.arange(n) + 0.3)
ax.set_xticklabels(assets)
ax.set_ylabel("Peso")
ax.set_title("Pesos óptimos: nominal vs. robusto (incertidumbre creciente)")
ax.legend()
plt.tight_layout()

### Interpretación

A medida que $\rho$ aumenta (mayor incertidumbre sobre $\boldsymbol{\mu}$):
- Los pesos se vuelven más **diversificados** (convergen hacia pesos iguales)
- El término $-\rho\|\mathbf{w}\|_2$ penaliza la concentración
- Con $\rho \to \infty$, el portafolio robusto converge al de **mínima varianza**

---
## 3. Problema de dieta con Pyomo (MILP)

### ¿Por qué Pyomo y no CVXPY?

El problema de dieta tiene **variables enteras** ($x_i \in \mathbb{Z}_{\geq 0}$: número de porciones). Los problemas con variables enteras son **NP-hard** (Boyd §4.1.3) y requieren algoritmos especializados (branch-and-bound) que Pyomo maneja con solvers como GLPK.

### Formulación

$$
\min_{\mathbf{x}} \quad \mathbf{c}^\top \mathbf{x}
$$

sujeto a:

$$
\mathbf{N}^{\min} \leq A\mathbf{x} \leq \mathbf{N}^{\max}, \qquad \sum_i V_i x_i \leq V^{\max}, \qquad x_i \in \mathbb{Z}_{\geq 0}
$$

donde $c_i$ es el costo por porción, $A_{ij}$ es el contenido del nutriente $j$ en el alimento $i$, y $V_i$ es el volumen por porción.

### Modelo Pyomo (`diet.py`)

El modelo está definido en `diet.py` como un `AbstractModel` de Pyomo con datos en `dietdata.dat`. Los componentes son:
- **Conjuntos**: alimentos ($F$), nutrientes ($N$)
- **Parámetros**: costos, contenido nutricional, límites
- **Variables**: $x_i$ = porciones de alimento $i$ (enteros no negativos)
- **Objetivo**: minimizar costo total
- **Restricciones**: nutrientes dentro de rango, volumen máximo

In [ ]:
# Mostrar el modelo Pyomo
print("=== Modelo Pyomo (diet.py) ===\n")
with open("diet.py") as f:
    print(f.read())

In [ ]:
# Mostrar los datos
print("=== Datos (dietdata.dat) ===\n")
with open("dietdata.dat") as f:
    print(f.read())

### Resolución con Pyomo + GLPK

> **Nota**: GLPK debe estar instalado en el sistema. En Colab: `!apt-get install -q glpk-utils`. En Binder, puede no estar disponible — en ese caso el resultado se muestra desde `results.yml`.

In [ ]:
import shutil

if shutil.which("glpsol"):
    # GLPK disponible: resolver
    import subprocess
    result = subprocess.run(
        ["pyomo", "solve", "--solver=glpk", "diet.py", "dietdata.dat"],
        capture_output=True, text=True, cwd="."
    )
    print(result.stdout)
    if result.returncode != 0:
        print("Error:", result.stderr)
else:
    print("GLPK no disponible. Mostrando resultados previos:\n")
    with open("results.yml") as f:
        print(f.read())

---
## 4. Comparación: CVXPY vs. Pyomo

### Resumen del curso

| Problema | Herramienta | Clase | Tipo |
|----------|-------------|:-----:|------|
| Frontera eficiente (Markowitz) | CVXPY | 4, 11 | QP |
| Portafolio con restricción de riesgo | CVXPY | 15 | QCQP |
| Portafolio robusto | CVXPY | 15 | SOCP |
| Maximización de Sharpe (Charnes-Cooper) | CVXPY | 4, 9, 10 | QP |
| Regularización L₂/L₁ | CVXPY | 12 | QP |
| Tracking error | CVXPY | 12 | SOCP |
| Valuación de opciones barrera | Monte Carlo | 14 | No convexo |
| Problema de dieta (variables enteras) | Pyomo | 15 | MILP |

### Flujo completo del curso

```
Datos (yfinance) → Rendimientos (log)
  → Estimadores robustos (Huber μ, Ledoit-Wolf Σ)
    → Simulación Monte Carlo (portafolios, opciones, barrera)
    → Optimización convexa (CVXPY/DCP)
      → QP: frontera eficiente
      → QCQP: restricción de riesgo
      → SOCP: tracking error, robusta, CML
    → Programación entera (Pyomo)
      → MILP: problema de dieta
```

---

## Navegación del curso

← **Anterior**: Clase 14: Opciones barrera  
→ **Fin del curso**

---

## 5. Referencias bibliográficas

- **Birge, J. R. & Louveaux, F.** (2011). *Introduction to Stochastic Programming* (2nd ed.). Springer.
- **Boyd, S. & Vandenberghe, L.** (2004). *Convex Optimization*. Cambridge University Press. — §4.1.3 (convexo vs. entero), §4.4.2 (QCQP), §4.6 (SOCP), §6.5 (multi-objetivo), §7.1 (incertidumbre). [PDF gratuito](https://web.stanford.edu/~boyd/cvxbook/).
- **Charnes, A. & Cooper, W. W.** (1962). Programming with linear fractional functionals. *Naval Research Logistics Quarterly*, 9(3–4), 181–186.
- **Glasserman, P.** (2003). *Monte Carlo Methods in Financial Engineering*. Springer.
- **Hull, J. C.** (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson.
- **Ledoit, O. & Wolf, M.** (2004). A well-conditioned estimator for large-dimensional covariance matrices. *Journal of Multivariate Analysis*, 88(2), 365–411.
- **Luenberger, D. G.** (2013). *Investment Science* (2nd ed.). Oxford University Press.
- **Markowitz, H.** (1952). Portfolio Selection. *The Journal of Finance*, 7(1), 77–91.
- **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley.
- **Venegas Martínez, F.** (2008). *Riesgos financieros y económicos* (2a ed.). Cengage Learning.